# Week 9-14 Coding Quizzes
## DX701 Experimental Design and Causality
### Alyssa Player

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


# Week 9

In [ ]:
def simulate(A=1, B=1, C=10, D=1000):
  W = np.random.normal(0,1,D)
  X = W+np.random.normal(0,B,D)
  Y = A*X-W+np.random.normal(0,C,D)
  return Y, X, W

## Question 1
Which of the following is closest to the probability of detecting a nonzero effect of X on Y (the t-value of X is greater in absolute value than about 1.96) given A = 1, B = 1, C = 10, D = 1000? Include W in the regression.

In [ ]:
import statsmodels.api as sm

def run_sim(n_sims=5000, A=1, B=1, C=10, D=1000):
    sig = 0
    results = []
    for i in range(n_sims):
        Y, X, W = simulate(A=A, B=B, C=C, D=D)
        X_matrix = sm.add_constant(np.column_stack((X, W)))
        model = sm.OLS(Y, X_matrix).fit()
        t_val = model.tvalues[1]          # fix 1: index into tvalues, not call it; [1] = X's t-value
        if abs(t_val) > 1.96:              # fix 2 & 3: matched variable name + defined threshold directly
            sig += 1
    return sig / n_sims                    # fix 4: dedented so it's outside the for loop

power = run_sim()
print(power)

0.8742


## Question 2
Which of the following is closest to the skew of the estimate in that case? (You can compute this using scipy.)

In [ ]:
from scipy.stats import skew

def run_sim(n_sims=5000, A=1, B=1, C=10, D=1000):
    sig = 0
    estimates = []  # updated line to get the skew of the est.
    for i in range(n_sims):
        Y, X, W = simulate(A=A, B=B, C=C, D=D)
        X_matrix = sm.add_constant(np.column_stack((X, W)))
        model = sm.OLS(Y, X_matrix).fit()
        t_val = model.tvalues[1]
        estimates.append(model.params[1])   # updated line so x can estimate
        if abs(t_val) > 1.96:
            sig += 1
    power = sig / n_sims
    return power, np.array(estimates)

power, estimates = run_sim()
print("power:", power)
print("skew:", skew(estimates))

power: 0.8898
skew: -0.03256392021191451


## Question 3:
With A = 1, C = 10, D = 1,000, what value of B is needed to detect that the Data Generating Process (DGP) has a nonzero coefficient for X about 50% of the time? (Choose the closest value.)


In [ ]:
for B in [1.8, 0.6, 0.2, 5.4]:
    power, _ = run_sim(B=B)
    print(B, "=", power)

1.8 = 0.9992
0.6 = 0.4674
0.2 = 0.0908
5.4 = 1.0


## Question 4:
Question 4
With B = 1, C = 10, D = 100 (note the different value of D), what value of A is needed to detect that the DGP has a nonzero coefficient for X about 50% of the time? (Choose the closest value.)

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import skew

def run_sim_q4(n_sims=5000, A=1, B=1, C=10, D=100):
    sig = 0
    estimates = []
    for i in range(n_sims):
        Y, X, W = simulate(A=A, B=B, C=C, D=D)   # fixed: A, not A_q4
        X_matrix = sm.add_constant(np.column_stack((X, W)))
        model = sm.OLS(Y, X_matrix).fit()
        t_val = model.tvalues[1]
        estimates.append(model.params[1])
        if abs(t_val) > 1.96:
            sig += 1
    power = sig / n_sims
    return power, np.array(estimates)

for A in [2.0, 0.5, 4.0, 1.0]:
    power, _ = run_sim_q4(A=A)
    print(A, "=", power)


2.0 = 0.516
0.5 = 0.0818
4.0 = 0.9698
1.0 = 0.1744


# Week 10

## Question 1:
Use the data in homework_10.1.csv and find the fixed effect (the constant term in the regression) for each time (0 through 11). Which of these describes the pattern:

In [ ]:
import pandas as pd
df = pd.read_csv("homework_10.1.csv")
df.head()

,Unnamed: 0,city,time,X,y
0,0,0,0,0.144044,7.552716
1,1,0,1,1.454274,10.077829
2,2,0,2,0.761038,12.372731
3,3,0,3,0.121675,11.489263
4,4,0,4,0.443863,13.104833


In [ ]:
#time should be categorical
import statsmodels.formula.api as smf

model = smf.ols("y ~ C(time)", data=df).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.175
Model:                            OLS   Adj. R-squared:                  0.149
Method:                 Least Squares   F-statistic:                     6.711
Date:                Sun, 26 Jul 2026   Prob (F-statistic):           3.19e-10
Time:                        11:18:07   Log-Likelihood:                -1027.5
No. Observations:                 360   AIC:                             2079.
Df Residuals:                     348   BIC:                             2126.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         2.5605      0.780      3.282

## Question 2:
What about the fixed effect for each city (0 through 9)?



In [ ]:
#time should be categorical
import statsmodels.formula.api as smf

model = smf.ols("y ~ C(city)", data=df).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.691
Model:                            OLS   Adj. R-squared:                  0.683
Method:                 Least Squares   F-statistic:                     86.80
Date:                Sun, 26 Jul 2026   Prob (F-statistic):           1.23e-83
Time:                        11:20:45   Log-Likelihood:                -850.99
No. Observations:                 360   AIC:                             1722.
Df Residuals:                     350   BIC:                             1761.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       10.8712      0.435     25.000   

## Question 3:

For the following data, model np.exp(Y) as a function of X and Z.

num = 10000
X = np.clip(np.random.normal(3, 1, (num,)), 0.01, 100)
Z = np.clip(np.random.normal(3, 1, (num,)), 0.01, 100)
Y = np.log(X + Z) + np.random.normal(0, 1, (num,))

With enough data, the coefficients of X and Z are closest to:

In [ ]:
import statsmodels.api as sm
num = 10000
X = np.clip(np.random.normal(3, 1, (num,)), 0.01, 100)
Z = np.clip(np.random.normal(3, 1, (num,)), 0.01, 100)
Y = np.log(X + Z) + np.random.normal(0, 1, (num,))

expY = np.exp(Y)
Xmat = sm.add_constant(np.column_stack([X, Z]))
model = sm.OLS(expY, Xmat).fit()
print(model.params)

[-0.41715294  1.62620564  1.89519314]


## Question 4:

Suppose we were to use the data below to find the standard error of X's coefficient (the coefficient that should be 1.5) in two ways:

1)  By asking Python to find the standard error.

2)  By simulating what happens if we generate the data 100 times, estimating X's coefficient and finding the standard deviation of the 100 estimates, using this model:



     num = 10000

     Z = np.random.normal(0, 1, (num,))

     X = Z + np.random.normal(0, 1, (num,))

     Y = 1.5 * X + 2.3 * Z + np.random.normal(0, X**2, (num,))

           

 Then:

In [ ]:
#asking python
import numpy as np
import statsmodels.api as sm

num = 10000
Z = np.random.normal(0, 1, (num,))
X = Z + np.random.normal(0, 1, (num,))
Y = 1.5 * X + 2.3 * Z + np.random.normal(0, X**2, (num,))

Xmat = sm.add_constant(np.column_stack([X, Z]))
model = sm.OLS(Y, Xmat).fit()

print("Coefficients =", model.params)
print("Standard Errors =", model.bse)

Coefficients = [0.01480273 1.40052541 2.37367308]
Standard Errors = [0.0358147  0.03594511 0.05109907]


In [ ]:
#simulation
coefs = []

for i in range(100):
    Z = np.random.normal(0, 1, (num,))
    X = Z + np.random.normal(0, 1, (num,))
    Y = 1.5 * X + 2.3 * Z + np.random.normal(0, X**2, (num,))

    Xmat = sm.add_constant(np.column_stack([X, Z]))
    model = sm.OLS(Y, Xmat).fit()
    coefs.append(model.params[1])

coefs = np.array(coefs)
print("SE:", coefs.std())

SE: 0.05358580316951777


# Week 11

In [ ]:
import statsmodels.api as sm

num = 1000

event_time = int(num / 2)

R_market = np.random.normal(0, 1, num) + np.arange(num) / num

R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2

results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()

alpha, beta = results.params

resid = R_target - results.predict(sm.add_constant(R_market))

print(resid[event_time + 1] / resid[:event_time].std(ddof = 2))

0.46988451480057386


##Question 1:
Which is closest to the probability that this t-test will be able to detect the event at event_time + 1 with the given code?

In [ ]:
#0.5 was the answer

## Question 2:

Use the same code but put np.random.seed(0) at the beginning of each loop to ensure that you are performing placebo tests on a fixed dataset. Perform a placebo test by setting the fictitious event_time to all possible times, while leaving the event in R_target at just the 1 time. The placebo test trains itself on the data leading up to the fictitious event. About what fraction of placebo tests seem to detect an event at the fictitious event time?



## Question 3:

Do the same placebo test, but this time only run the test 20 times before and twenty times after the actual event. On average (over many runs of the code), what fraction of the 40 placebo tests get a higher t-value than the actual event? This time, adjust np.random.seed() to represent a different dataset when needed.



In [ ]:
import numpy as np
import statsmodels.api as sm

num = 1000
event_time = int(num / 2)
threshold = 1.96
detections = 0
total = 0

for t in range(2, num - 1):
    np.random.seed(0)
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == event_time + 1) * 2

    results = sm.OLS(R_target[:t], sm.add_constant(R_market[:t])).fit()
    resid = R_target - results.predict(sm.add_constant(R_market))
    t_stat = resid[t + 1] / resid[:t].std(ddof=2)

    if abs(t_stat) > threshold:
        detections += 1
    total += 1

print(detections / total)

0.04714142427281846


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: divide by zero encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
import numpy as np
import statsmodels.api as sm

num = 1000
event_time = int(num / 2)
threshold = 1.96
detections = 0
total = 0

for t in range(2, num - 1):
    np.random.seed(0)
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == event_time + 1) * 2

    results = sm.OLS(R_target[:t], sm.add_constant(R_market[:t])).fit()
    resid = R_target - results.predict(sm.add_constant(R_market))
    t_stat = resid[t + 1] / resid[:t].std(ddof=2)

    if abs(t_stat) > threshold:
        detections += 1
    total += 1

print(detections / total)  # ~0.05

0.04714142427281846


In [ ]:
def make_error(corr_const, num):


  sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2))


  err = list()


  prev = np.random.normal(0, sigma)


  for n in range(num):


    prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma)


    err.append(prev)


  return np.array(err)

## Question 4:
Do the same thing as in question 2, but this time use make_error with corr_const = 0.9 to generate the error for R_target instead of np.random.normal. Consider before attempting this: Would you expect this kind of dataset, where errors are not independent, to result in more or fewer false positives in the placebo tests?



In [ ]:
import numpy as np
import statsmodels.api as sm

def make_error(corr_const, num):
    sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2))
    err = list()
    prev = np.random.normal(0, sigma)
    for n in range(num):
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma)
        err.append(prev)
    return np.array(err)

num = 1000
event_time = int(num / 2)
threshold = 1.96
detections = 0
total = 0

for t in range(2, num - 1):
    np.random.seed(0)
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = 2 + R_market + make_error(0.9, num) + (np.arange(num) == event_time + 1) * 2

    results = sm.OLS(R_target[:t], sm.add_constant(R_market[:t])).fit()
    resid = R_target - results.predict(sm.add_constant(R_market))
    t_stat = resid[t + 1] / resid[:t].std(ddof=2)

    if abs(t_stat) > threshold:
        detections += 1
    total += 1

print(detections / total)

0.04714142427281846


#Week 12

In [3]:
df_12_1 = pd.read_csv("homework_12.1.csv")
df_12_2 = pd.read_csv("homework_12.2.csv")

## Question 1

Find the effect in dataset 12.1,
given that Group = 1 is the treatment group and Time > 0 is the treatment time.

- coefficient of group:post interaction ~1 (0.944)


In [4]:
df_12_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  10000 non-null  int64  
 1   Y           10000 non-null  float64
 2   Time        10000 non-null  float64
 3   Group       10000 non-null  int64  
dtypes: float64(2), int64(2)
memory usage: 312.6 KB


In [5]:
import statsmodels.formula.api as smf

df_12_1['Post'] = (df_12_1['Time'] > 0).astype(int)

model = smf.ols('Y ~ Group * Post', data=df_12_1).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.638
Model:                            OLS   Adj. R-squared:                  0.638
Method:                 Least Squares   F-statistic:                     5866.
Date:                Thu, 06 Aug 2026   Prob (F-statistic):               0.00
Time:                        17:34:57   Log-Likelihood:                -14214.
No. Observations:               10000   AIC:                         2.844e+04
Df Residuals:                    9996   BIC:                         2.846e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9802      0.016     61.953      0.0

## Question 2

With dataset 12.2, run a test to check prior trends. To do this, run a linear regression on the data before Time = 0, with an interaction term equal to Group x Time. Which is the closest to the t-value of the interaction term?

- significance of the group:time interaction

In [6]:
import statsmodels.formula.api as smf

# pre-treatment period only
pre = df_12_2[df_12_2['Time'] < 0]

model = smf.ols('Y ~ Group * Time', data=pre).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.324
Model:                            OLS   Adj. R-squared:                  0.324
Method:                 Least Squares   F-statistic:                     1591.
Date:                Thu, 06 Aug 2026   Prob (F-statistic):               0.00
Time:                        17:38:10   Log-Likelihood:                -13981.
No. Observations:                9944   AIC:                         2.797e+04
Df Residuals:                    9940   BIC:                         2.800e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9996      0.022     45.154      0.0

## Question 3

With the following, make an error such that the correlation (np.corrcoef) between the error at time t and time t + 1 is 0.80 on average, the mean error is 0, and the standard deviation of the error is 1. (These may be approximate.) There are 10,000 items. Given:

X = this error

Y = 2 * X + this error (recalculated with new randomness relative to the previous line)

**The standard error of the estimated X coefficient computed via Ordinary Least Squares (OLS) basic standard error should be about 0.01. Then the standard error computed via simulation (the standard deviation of the estimated coefficient over many trials) is closest to:**

Note: you might consider Googling / asking ChatGPT how to generate an autocorrelated time series with a specific correlation and standard deviation.

In [13]:
import numpy as np
import statsmodels.api as sm

def make_error(n, rho=0.8, sigma=1.0, rng=None):
    """AR(1) process with mean 0, stationary std = sigma, lag-1 corr = rho."""
    innov_sd = sigma * np.sqrt(1 - rho**2)
    e = np.zeros(n)
    e[0] = rng.normal(0, sigma)
    for t in range(1, n):
        e[t] = rho * e[t-1] + rng.normal(0, innov_sd)
    return e

n = 10000
rng = np.random.default_rng(0)

X = make_error(n, rho=0.8, sigma=1.0, rng=rng)
eps_Y = make_error(n, rho=0.8, sigma=1.0, rng=rng)
Y = 2 * X + eps_Y



In [14]:
Xc = sm.add_constant(X)
model = sm.OLS(Y, Xc).fit()
print("Basic se of coef:", model.bse[1])

Basic se of coef: 0.0097775229026573


In [15]:
#stdev from many trials
n_trials = 500
coefs = np.zeros(n_trials)

for i in range(n_trials):
    Xi = make_error(n, rho=0.8, sigma=1.0, rng=rng)
    eps_Yi = make_error(n, rho=0.8, sigma=1.0, rng=rng)
    Yi = 2 * Xi + eps_Yi

    Xic = sm.add_constant(Xi)
    m = sm.OLS(Yi, Xic).fit()
    coefs[i] = m.params[1]

print("Simulated SE across trials:", coefs.std())

Simulated SE across trials: 0.021831106721850298


## Question 4:

Find the variance inflation factor of X1:

np.random.seed(0)

X1 = np.random.normal(0, 1, 1000)

X2 = np.random.normal(0, 1, 1000) + X1

X3 = np.random.normal(0, 1, 1000) + 2 * X2

In [11]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

np.random.seed(0)
X1 = np.random.normal(0, 1, 1000)
X2 = np.random.normal(0, 1, 1000) + X1
X3 = np.random.normal(0, 1, 1000) + 2 * X2

df = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3})
X = sm.add_constant(df)

var_if_X1 = variance_inflation_factor(X.values, 1)  # column index 1 is position of x1
print(var_if_X1)

1.9783742480901254
